In [160]:
import re  #regex
import pandas as pd
import nltk  #natural language toolkit
from nltk.corpus import stopwords  #stopword are words adding insignificant meaning
from nltk.stem import WordNetLemmatizer  # to reduce words to their base word... Running wil be run, dresses,.... dress... bought... buy
import pandas as pd
import numpy as np
import string
from textblob import TextBlob
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import GaussianNB

In [161]:
pip install textblob

Note: you may need to restart the kernel to use updated packages.


In [162]:
# let download require resources from nltk
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [163]:
df = pd.read_csv(r"C:\Users\PC\Desktop\data_amazon.csv")

In [164]:
df.head(2)

,Title,Review,Cons_rating,Cloth_class,Materials,Construction,Color,Finishing,Durability
0,NaN,Absolutely wonderful - silky and sexy and comf...,4.0,Intimates,0.0,0.0,0.0,1.0,0.0
1,NaN,Love this dress! it's sooo pretty. i happene...,5.0,Dresses,0.0,1.0,0.0,0.0,0.0


In [165]:
df['Cloth_class'].unique()

array(['Intimates', 'Dresses', 'Pants', 'Blouses', 'Knits', 'Outerwear',
       'Lounge', 'Sweaters', 'Skirts', 'Fine gauge', 'Sleep', 'Jackets',
       'Swim', 'Trend', 'Jeans', 'Legwear', 'Shorts', 'Layering',
       'Casual bottoms', nan, 'Chemises', 'Blazer'], dtype=object)

In [166]:
df.drop(['Title','Cons_rating','Cloth_class','Materials','Construction','Color','Finishing','Durability'], axis = 1 , inplace = True)

In [167]:
df.head()

,Review
0,Absolutely wonderful - silky and sexy and comf...
1,Love this dress! it's sooo pretty. i happene...
2,I had such high hopes for this dress and reall...
3,"I love, love, love this jumpsuit. it's fun, fl..."
4,This shirt is very flattering to all due to th...


In [168]:
df.isnull().sum()

Review    805
dtype: int64

In [169]:
df.isnull().sum().sum()

np.int64(805)

In [170]:
df.dropna( inplace = True)

In [171]:
df.tail(10)

,Review
26519,Itâ€™s summer perfect!
26520,I have several of these shirts and get complim...
26521,My daughter loves this top.
26522,The sleeves are to long and the neck line was ...
26523,The fit is great. I am a large lady and this ...
26524,"Not the right color, fit was way too small, n..."
26525,Slightly different fabric than pictured but Iâ...
26526,Very cute top nice and lightweight but the sle...
26527,Love this shirt so much. Very good quality and...
26528,"Nice flowy fabric, pretty mauve color, I feel ..."


In [172]:
df.shape

(25724, 1)

In [173]:
def get_polarity(text):
    return TextBlob(str(text)).sentiment.polarity

In [176]:
df['Polarity']= df['Review'].apply(get_polarity)

In [177]:
df.head()

,Review,Polarity
0,Absolutely wonderful - silky and sexy and comf...,0.633333
1,Love this dress! it's sooo pretty. i happene...,0.339583
2,I had such high hopes for this dress and reall...,0.073675
3,"I love, love, love this jumpsuit. it's fun, fl...",0.550000
4,This shirt is very flattering to all due to th...,0.512891


In [178]:
# convert or rename the Polarity col to sentiment labels
def polarity_sentiment(score):
    if score > 0: 
        return "Positive"
    elif score <0:
        return "Negative"
    else:
        return "Neutral"

In [179]:
df['Sentiment']= df['Polarity'].apply(polarity_sentiment)

In [180]:
df.head()

,Review,Polarity,Sentiment
0,Absolutely wonderful - silky and sexy and comf...,0.633333,Positive
1,Love this dress! it's sooo pretty. i happene...,0.339583,Positive
2,I had such high hopes for this dress and reall...,0.073675,Positive
3,"I love, love, love this jumpsuit. it's fun, fl...",0.550000,Positive
4,This shirt is very flattering to all due to th...,0.512891,Positive


In [181]:
#stopwords = set(stopwords.words('english'))
#lemmetizer = WordNetLemmatizer()

In [182]:
# clean the review using RE
def clean_text(text):
    text = text.lower()                          # convert to lowercase
    text = re.sub(r'http\S+', '', text)          # remove URLs
    text = re.sub(r'@\w+', '', text)             # remove mentions
    text = re.sub(r'#\w+', '', text)             # remove hashtags
    text = re.sub(r'\d+', '', text)              # remove numbers
    text = re.sub(r"[-'.]", '', text)            # remove hyphen, full stop, apostrophe
    text = re.sub(r'[^\w\s]', '', text)          # remove remaining punctuation
    text = re.sub(r'\s+', ' ', text)             # keep only ONE space between words
    text = text.strip()                          # remove leading/trailing spaces
    
    return text

In [183]:
df['CleanReview']= df['Review'].apply(clean_text)

In [184]:
df.head()

,Review,Polarity,Sentiment,CleanReview
0,Absolutely wonderful - silky and sexy and comf...,0.633333,Positive,absolutely wonderful silky and sexy and comfor...
1,Love this dress! it's sooo pretty. i happene...,0.339583,Positive,love this dress its sooo pretty i happened to ...
2,I had such high hopes for this dress and reall...,0.073675,Positive,i had such high hopes for this dress and reall...
3,"I love, love, love this jumpsuit. it's fun, fl...",0.550000,Positive,i love love love this jumpsuit its fun flirty ...
4,This shirt is very flattering to all due to th...,0.512891,Positive,this shirt is very flattering to all due to th...


In [185]:
x = df['CleanReview']
y = df['Sentiment']

In [186]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,random_state=42, test_size = 0.2)

In [187]:
vectorizer = TfidfVectorizer(max_features=500,stop_words='english')

In [188]:
xtrain_vec = vectorizer.fit_transform(xtrain)

In [189]:
xtestvec=vectorizer.transform(xtest)

In [190]:
log = LogisticRegression()

In [191]:
log.fit(xtrain_vec, ytrain)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [192]:
logpred = log.predict(xtestvec)

In [193]:
accuracy= accuracy_score(logpred, ytest)*100

In [194]:
accuracy

93.29446064139941

In [195]:
# let test some reviews
def predict_review(text):
    clean= clean_text(text)
    vectorized=vectorizer.transform([clean])
    prediction = log.predict(vectorizer)[0]
    return prediction

In [196]:
#df.predict_review("this is a bad cloth i dont like it")

In [197]:
dtc = DecisionTreeClassifier()

In [198]:
dtc.fit(xtrain_vec,ytrain)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [200]:
dtcpred = dtc.predict(xtestvec)

In [201]:
dtcpred[:30]

array(['Positive', 'Positive', 'Positive', 'Positive', 'Positive',
       'Negative', 'Positive', 'Positive', 'Positive', 'Positive',
       'Positive', 'Negative', 'Positive', 'Positive', 'Positive',
       'Positive', 'Positive', 'Negative', 'Positive', 'Positive',
       'Positive', 'Positive', 'Positive', 'Positive', 'Positive',
       'Positive', 'Positive', 'Positive', 'Positive', 'Positive'],
      dtype=object)

In [202]:
dtc_accuracy = accuracy_score(dtcpred,ytest)*100

In [203]:
dtc_accuracy

89.52380952380953

In [204]:
naive = GaussianNB()

In [216]:
xtrain = xtrain_vec.toarray()
xtest = xtestvec.toarray()

In [217]:
naive.fit(xtrain,ytrain)

,priors,None
,var_smoothing,1e-09


In [218]:
naive_pred = dtc.predict(xtest)

In [219]:
naive_accuracy = accuracy_score(dtcpred, ytest)*100

In [220]:
naive_accuracy

89.52380952380953